# 💬 Prática 3 — Validador & Scanner do domínio: **Chatbot de atendimento (estilo WhatsApp)**

**Disciplina:** Compiladores — Aula CP 08 (Prática 3) · **Tema:** Chatbot de atendimento

### O que este notebook entrega

| Req. | O que é | Onde |
|---|---|---|
| R1 | Dois validadores (AFD): **CPF formatado** (15 estados) e **protocolo** `AAAA-NNNNNN` (12 estados, com classes "finas") | Células 1 e 2 |
| R2 | Rastreamento passo a passo (`rodar()`) com palavras aceitas e rejeitadas | Células 1 e 2 |
| R3 | Bateria AFD × RegEx com 0 divergências (inclui palavra vazia e um teste automático com milhares de palavras) | Célula 3 |
| R4 | Máquina de estados da conversa (9 estados, 31 transições), com eventos permitidos nas recusas | Célula 4 |
| R5 | Mini-scanner de roteiros do chatbot (8 tipos de token, 10 palavras reservadas, Maximal Munch, erros com dica) | Células 5 e 6 |
| R6 | Interface com `ipywidgets` para os validadores e o scanner | Célula 7 |

> Execute as células **em ordem** (`Ambiente de execução → Executar tudo`). Só a Célula 7 usa `ipywidgets`.

In [ ]:
# CÉLULA 0 — Preparação do ambiente
# Tudo usa só a biblioteca padrão do Python (+ ipywidgets, só na interface da Célula 7).
# Funciona no Google Colab e no VS Code (Jupyter). No VS Code: pip install ipywidgets
import re, random, string, difflib
import ipywidgets as widgets
from IPython.display import display

LETRAS  = set(string.ascii_letters + "_")      # letras ASCII e "_"
DIGITOS = set("0123456789")                    # só dígitos ASCII
ALNUM   = LETRAS | DIGITOS

print("✅ Ambiente pronto! Vamos construir autômatos para o chatbot.")

## O "motor": a classe `AFD`

A mesma classe da aula (`M = (Q, Σ, δ, q0, F)`), com três poderes extras que usaremos no projeto:

* `tabela()` agora ajusta a largura das colunas e ordena `q2` antes de `q10`;
* `aceita(palavra)` devolve só `True/False`;
* `maior_prefixo(texto, i)` devolve o fim do **maior** prefixo aceito (é o **Maximal Munch** do scanner);

In [ ]:
# CÉLULA — A classe AFD (o "motor" reutilizável da aula)

def chave_natural(estado):
    """Ordem natural: 'q2' vem antes de 'q10'."""
    return [int(p) if p.isdigit() else p for p in re.split(r"(\d+)", estado)]

class AFD:
    def __init__(self, nome, delta, inicial, finais, classificar):
        self.nome = nome
        self.delta = delta                      # δ: {(estado, classe): próximo}
        self.inicial = inicial                  # q0
        self.finais = set(finais)               # F
        self.classificar = classificar          # função: caractere -> classe (Σ)
        self.estados = sorted({e for e, _ in delta} | set(delta.values()) | {inicial}, key=chave_natural)   # Q
        self.alfabeto = sorted({s for _, s in delta})                                                       # Σ

    def rodar(self, palavra, mostrar=True):
        estado = self.inicial
        larg = max(len(q) for q in self.estados)
        if mostrar:
            print(f"🔎 {self.nome} lendo {palavra!r}")
            print(f"   {'passo':5} {'lido':8} {'classe':8} {'de':{larg}}  para")
        for passo, c in enumerate(palavra, start=1):
            classe = self.classificar(c)
            proximo = self.delta.get((estado, classe))
            if mostrar:
                print(f"   {passo:<5} {c!r:8} {str(classe):8} {estado:{larg}}  {proximo or '💀 morto'}")
            if proximo is None:
                if mostrar:
                    print(f"   ❌ REJEITADA: não há transição de {estado} com {classe!r} (morreu no meio)\n")
                return False
            estado = proximo
        aceita = estado in self.finais
        if mostrar:
            if aceita:
                print(f"   ✅ ACEITA: terminou em {estado}, que é de aceitação\n")
            else:
                print(f"   ❌ REJEITADA: terminou em {estado}, que NÃO é de aceitação (faltou texto)\n")
        return aceita

    def aceita(self, palavra):
        return self.rodar(palavra, mostrar=False)

    def maior_prefixo(self, texto, inicio=0):
        """Maximal Munch: fim (exclusivo) do MAIOR prefixo de texto[inicio:] que o AFD aceita (ou None)."""
        estado, melhor, i = self.inicial, None, inicio
        if estado in self.finais:
            melhor = inicio
        while i < len(texto):
            proximo = self.delta.get((estado, self.classificar(texto[i])))
            if proximo is None:
                break
            estado, i = proximo, i + 1
            if estado in self.finais:
                melhor = i
        return melhor

    def tabela(self):
        """Tabela de transição: linhas = estados, colunas = classes."""
        lq = max(len(q) for q in self.estados)
        larg = {s: max([len(str(s))] + [len(self.delta.get((q, s), "—")) for q in self.estados]) + 2
                for s in self.alfabeto}
        print(f"📋 Tabela de transição — {self.nome}")
        print("   " + f"{'estado':{lq}}" + "".join(f"{str(s):>{larg[s]}}" for s in self.alfabeto))
        for q in self.estados:
            marca = ("→" if q == self.inicial else " ") + ("*" if q in self.finais else " ")
            linha = "".join(f"{self.delta.get((q, s), '—'):>{larg[s]}}" for s in self.alfabeto)
            print(f"{marca} {q:{lq}}{linha}")
        print("   (→ = inicial, * = aceitação, — = vai para o estado morto)\n")

print("✅ Classe AFD pronta.")

## Validador 1 — CPF formatado `000.000.000-00` (R1 e R2)

Só o **formato** (11 dígitos com pontos e hífen), não os dígitos verificadores. As classes são `D` (dígito), `.` e `-`. Desenho:

```
──► (c0) ─D─► (c1) ─D─► (c2) ─D─► (c3) ─.─► (c4) ─D─► (c5) ─D─► (c6) ─D─► (c7) ─.─►
    (c8) ─D─► (c9) ─D─► (c10) ─D─► (c11) ─-─► (c12) ─D─► (c13) ─D─► ((c14))
```

Duas formas de rejeitar: **morrer no meio** (`123.45a...`) e **terminar fora de F** (`123.456.789-0`, faltou um dígito).

In [ ]:
# CÉLULA 1 — Validador 1: CPF formatado (000.000.000-00)
# Σ = {D, ., -}   D = qualquer dígito

def classe_cpf(c):
    if c in DIGITOS: return "D"
    if c == ".":     return "."
    if c == "-":     return "-"
    return "outro"

# Desenho copiado para o delta: o padrão "DDD.DDD.DDD-DD" gera as 14 setas c0 -> c1 -> ... -> c14
padrao_cpf = ["D"] * 3 + ["."] + ["D"] * 3 + ["."] + ["D"] * 3 + ["-"] + ["D"] * 2
delta_cpf = {(f"c{k}", s): f"c{k + 1}" for k, s in enumerate(padrao_cpf)}

afd_cpf = AFD("CPF formatado", delta_cpf, "c0", {"c14"}, classe_cpf)
afd_cpf.tabela()

afd_cpf.rodar("123.456.789-09")      # ✅ aceita
afd_cpf.rodar("123.45a.789-09")      # ❌ morre no meio (letra no lugar de dígito)
afd_cpf.rodar("123.456.789-0")       # ❌ termina em c13, que não é de aceitação (faltou 1 dígito)

## Validador 2 — Protocolo `AAAA-NNNNNN` (R1 e R2, com classes "finas")

Ano de **2000 a 2099** (`20` + 2 dígitos), hífen e 6 dígitos: `2026-000123`. O ano obriga a separar os dígitos em classes finas, como no horário da aula: o 1º dígito precisa ser `2`, o 2º precisa ser `0`, e os demais aceitam qualquer um.

| Classe | Caracteres |
|---|---|
| `0` | `0` |
| `2` | `2` |
| `1,3-9` | `1`, `3`, `4`, `5`, `6`, `7`, `8`, `9` |
| `-` | `-` |

```
──► (p0) ─2─► (p1) ─0─► (p2) ─qualquer─► (p3) ─qualquer─► (p4) ─-─► (p5)
    ─qualquer─► (p6) ─► (p7) ─► (p8) ─► (p9) ─► (p10) ─qualquer─► ((p11))
```
(*qualquer* = `0`, `2` ou `1,3-9`.)

In [ ]:
# CÉLULA 2 — Validador 2: protocolo de atendimento (AAAA-NNNNNN, ano 2000 a 2099)
# Classes "finas": o 1º dígito do ano é 2, o 2º é 0, o resto é "qualquer dígito".

def classe_protocolo(c):
    if c == "0":        return "0"
    if c == "2":        return "2"
    if c in DIGITOS:    return "1,3-9"          # os outros 8 dígitos
    if c == "-":        return "-"
    return "outro"

QUALQUER_DIGITO = ["0", "2", "1,3-9"]

delta_protocolo = {("p0", "2"): "p1", ("p1", "0"): "p2"}          # ano começa com "20"
for d in QUALQUER_DIGITO:
    delta_protocolo[("p2", d)] = "p3"                              # 3º dígito do ano: qualquer
    delta_protocolo[("p3", d)] = "p4"                              # 4º dígito do ano: qualquer
delta_protocolo[("p4", "-")] = "p5"                                # o hífen
for origem, destino in zip(["p5", "p6", "p7", "p8", "p9", "p10"], ["p6", "p7", "p8", "p9", "p10", "p11"]):
    for d in QUALQUER_DIGITO:
        delta_protocolo[(origem, d)] = destino                     # 6 dígitos do número

afd_protocolo = AFD("Protocolo AAAA-NNNNNN", delta_protocolo, "p0", {"p11"}, classe_protocolo)
afd_protocolo.tabela()

afd_protocolo.rodar("2026-000123")    # ✅ aceita
afd_protocolo.rodar("2126-000123")    # ❌ morre no meio: depois do 2 só cabe o 0 (o ano 21xx não existe)
afd_protocolo.rodar("2026-00012")     # ❌ termina em p10, que não é de aceitação (faltou 1 dígito)

## R3 — Bateria de testes AFD × RegEx

Cada AFD tem uma RegEx equivalente e precisam **concordar em tudo**. Usamos `re.fullmatch` (e não `re.search`) e incluímos a palavra vazia. Além das listas escritas à mão, o teste automático gera milhares de palavras aleatórias e, principalmente, **palavras quase corretas** (uma letra trocada, inserida ou removida), que é onde os bugs de desenho aparecem.

In [ ]:
# CÉLULA 3 — AFD × RegEx (eles têm que concordar!)

REGEX_CPF       = r"[0-9]{3}\.[0-9]{3}\.[0-9]{3}-[0-9]{2}"
REGEX_PROTOCOLO = r"20[0-9]{2}-[0-9]{6}"

ACEITAS_CPF = ["000.000.000-00", "123.456.789-09", "999.999.999-99", "111.222.333-44",
               "052.998.123-77", "321.654.987-10", "000.000.000-01"]
REJEITADAS_CPF = ["", "12345678909", "123.456.789-0", "123.456.789-099", "123.456.789 09",
                  "123.45.6789-09", "abc.def.ghi-jk", "123.456.789-0a", " 123.456.789-09",
                  "123.456.789-09 ", "123-456-789.09"]

ACEITAS_PROTOCOLO = ["2026-000123", "2000-000000", "2099-999999", "2024-123456",
                     "2026-000001", "2031-000999", "2010-100200"]
REJEITADAS_PROTOCOLO = ["", "2026000123", "1999-000123", "2100-000123", "2026-00012",
                        "2026-0001234", "2026-00012a", "20-000123", "2026-000123 ",
                        "abcd-000123", "2126-000123"]

def comparar(afd, regex, palavras, mostrar=True):
    """Roda cada palavra no AFD e na RegEx. Devolve o número de divergências."""
    divergencias = 0
    if mostrar:
        print(f"{'palavra':18} {'AFD':6} {'RegEx':6}")
    for p in palavras:
        r_afd = afd.aceita(p)
        r_re = re.fullmatch(regex, p) is not None
        divergencias += r_afd != r_re
        if mostrar:
            print(f"{p!r:18} {str(r_afd):6} {str(r_re):6} {'✅' if r_afd == r_re else '⚠️ DIVERGEM'}")
    return divergencias

def mutar(palavra, rnd, alfabeto):
    """Faz 1 ou 2 pequenas alterações (trocar, inserir ou apagar um caractere)."""
    for _ in range(rnd.choice([1, 1, 2])):
        k = rnd.randint(0, len(palavra))
        acao = rnd.choice(["trocar", "inserir", "apagar"])
        if acao == "inserir" or not palavra:
            palavra = palavra[:k] + rnd.choice(alfabeto) + palavra[k:]
        elif acao == "apagar":
            k = min(k, len(palavra) - 1)
            palavra = palavra[:k] + palavra[k + 1:]
        else:
            k = min(k, len(palavra) - 1)
            palavra = palavra[:k] + rnd.choice(alfabeto) + palavra[k + 1:]
    return palavra

def teste_automatico(afd, regex, aceitas, n=5000, semente=42):
    """Palavras aleatórias (50%) e palavras quase corretas (50%). Devolve (divergências, aceitas_por_ambos)."""
    rnd, alfabeto = random.Random(semente), "0123456789.-: aZ"
    divergencias = aceitas_ambos = 0
    for _ in range(n):
        if rnd.random() < 0.5:
            p = "".join(rnd.choice(alfabeto) for _ in range(rnd.randint(0, 16)))
        else:
            p = mutar(rnd.choice(aceitas), rnd, alfabeto)
        r_afd, r_re = afd.aceita(p), re.fullmatch(regex, p) is not None
        divergencias += r_afd != r_re
        aceitas_ambos += r_afd and r_re
    return divergencias, aceitas_ambos

BATERIAS = [("CPF formatado", afd_cpf, REGEX_CPF, ACEITAS_CPF, REJEITADAS_CPF),
            ("Protocolo", afd_protocolo, REGEX_PROTOCOLO, ACEITAS_PROTOCOLO, REJEITADAS_PROTOCOLO)]
DIVERGENCIAS_TOTAIS = 0
for nome, afd, regex, aceitas, rejeitadas in BATERIAS:
    print(f"━━━━ {nome}   RegEx: {regex}")
    d1 = comparar(afd, regex, aceitas + rejeitadas)
    print(f"→ {len(aceitas)} aceitas + {len(rejeitadas)} rejeitadas (incluindo a vazia) · divergências: {d1}")
    d2, ambos = teste_automatico(afd, regex, aceitas)
    print(f"→ teste automático: 5000 palavras ({ambos} aceitas por ambos) · divergências: {d2}\n")
    DIVERGENCIAS_TOTAIS += d1 + d2
print("Total de divergências:", DIVERGENCIAS_TOTAIS)

## R4 — Máquina de estados do negócio: a conversa do chatbot

Aqui os "símbolos" são **eventos** (`menu`, `opcao_1`, `cpf`...). O fluxo principal é `INICIO → MENU → SEGUNDA_VIA / FALAR_ATENDENTE → ENCERRADO`, com `voltar` e `ajuda`:

```
                  menu              opcao_1             cpf                 protocolo           pagar
──► (INICIO) ─────────► (MENU) ─────────► (SEGUNDA_VIA) ───► (AGUARDANDO_PROTOCOLO) ───► (BOLETO_ENVIADO) ───► (PAGAMENTO_CONFIRMADO)
                          │  ▲
                 opcao_2  │  │ voltar (de qualquer estado, menos INICIO e MENU)
                          ▼  │
                    (FALAR_ATENDENTE) ──msg (laço)──► (FALAR_ATENDENTE)
                          │ agendar
                          ▼
                    (ATENDIMENTO_AGENDADO)

 • sair  : qualquer estado ativo ──► ((ENCERRADO))      • ajuda : qualquer estado ativo ──► o mesmo estado
 • voltar: MENU ──► INICIO;  todos os outros ──► MENU
```

In [ ]:
# CÉLULA 4 — Máquina de estados do negócio: a conversa do chatbot
# Os "símbolos" são EVENTOS. É o mesmo modelo do pedido da aula (autômato fora do compilador).

ESTADOS_ATIVOS = ["INICIO", "MENU", "SEGUNDA_VIA", "AGUARDANDO_PROTOCOLO", "BOLETO_ENVIADO",
                  "PAGAMENTO_CONFIRMADO", "FALAR_ATENDENTE", "ATENDIMENTO_AGENDADO"]

delta_chat = {
    ("INICIO", "menu"): "MENU",
    ("MENU", "opcao_1"): "SEGUNDA_VIA",                       # 2ª via de boleto
    ("MENU", "opcao_2"): "FALAR_ATENDENTE",                   # falar com um atendente
    ("SEGUNDA_VIA", "cpf"): "AGUARDANDO_PROTOCOLO",           # identificou o cliente
    ("AGUARDANDO_PROTOCOLO", "protocolo"): "BOLETO_ENVIADO",  # informou o protocolo
    ("BOLETO_ENVIADO", "pagar"): "PAGAMENTO_CONFIRMADO",
    ("FALAR_ATENDENTE", "msg"): "FALAR_ATENDENTE",            # mandar mensagem (laço)
    ("FALAR_ATENDENTE", "agendar"): "ATENDIMENTO_AGENDADO",
}
for q in ESTADOS_ATIVOS:                                      # eventos que valem em qualquer estado ativo
    delta_chat[(q, "sair")] = "ENCERRADO"
    delta_chat[(q, "ajuda")] = q                              # pedir ajuda não muda de estado
    if q != "INICIO":
        delta_chat[(q, "voltar")] = "INICIO" if q == "MENU" else "MENU"

FINAIS_CHAT = {"ENCERRADO"}                                   # a conversa terminou
conversa = AFD("Conversa do chatbot", delta_chat, "INICIO", FINAIS_CHAT, lambda evento: evento)
print(f"{len(conversa.estados)} estados, {len(delta_chat)} transições\n")
conversa.tabela()

def processar_conversa(eventos):
    """Roda uma lista de eventos. Devolve 'finalizada', 'recusada' ou 'em andamento'."""
    estado = "INICIO"
    print(f"💬 eventos: {eventos}")
    for i, ev in enumerate(eventos, start=1):
        proximo = delta_chat.get((estado, ev))
        if proximo is None:                                   # evento recusado: mostra os PERMITIDOS
            permitidos = sorted(e for (q, e) in delta_chat if q == estado)
            lista = permitidos if permitidos else "nenhum (estado final: a conversa já terminou)"
            print(f"   ❌ evento {i} '{ev}' não é permitido em {estado}. Permitidos: {lista}\n")
            return "recusada"
        print(f"   {estado:22} --{ev}--> {proximo}")
        estado = proximo
    if estado in FINAIS_CHAT:
        print(f"   ✅ conversa finalizada em {estado}\n")
        return "finalizada"
    print(f"   ⏳ conversa ainda em andamento ({estado})\n")
    return "em andamento"

print("──────── 2 fluxos VÁLIDOS ────────")
processar_conversa(["menu", "opcao_1", "cpf", "protocolo", "pagar", "sair"])          # 2ª via até o pagamento
processar_conversa(["menu", "opcao_2", "msg", "agendar", "voltar", "ajuda", "sair"])  # atendente, agenda e volta
print("──────── 2 fluxos INVÁLIDOS ────────")
processar_conversa(["menu", "protocolo"])         # pulou etapas: protocolo no MENU
processar_conversa(["menu", "sair", "menu"])      # falou depois de encerrar
print("──────── e um incompleto ────────")
processar_conversa(["menu", "opcao_1"])           # parou no meio: não é erro, mas não terminou

## R5 — Mini-scanner para os roteiros do chatbot

Um **roteiro** é um texto com os comandos que o cliente digitaria:

```
# segunda via de boleto
/menu
/opcao 1
/cpf 123.456.789-09
/protocolo 2026-000123
/pagar 150.00
/sair
```

| Token | Padrão | Exemplos |
|---|---|---|
| `COMANDO` (palavras reservadas) | `/` + nome da lista: `menu opcao cpf protocolo agendar msg pagar ajuda voltar sair` | `/menu`, `/cpf` |
| `NUM_INT` | um ou mais dígitos | `2` |
| `VALOR` | dígitos, ponto e **2** dígitos | `150.00` |
| `CPF` | o AFD do validador 1 | `123.456.789-09` |
| `PROTOCOLO` | o AFD do validador 2 | `2026-000123` |
| `HORARIO` | o AFD do exemplo 4 da aula | `14:30` |
| `TEXTO` | qualquer coisa entre aspas duplas, na mesma linha | `"preciso de ajuda"` |
| `ID` | letra ou `_`, seguida de letras, dígitos ou `_` | `protocolo` (em `/ajuda protocolo`) |
| *(ignorados)* | espaços, quebras de linha e comentários `# ...` | |

**Maximal Munch (o caso do domínio):** `2026-000123` é **um** `PROTOCOLO`, não `2026` + `-` + `000123`; `14:30` é um `HORARIO`, não `14` + `:` + `30`; `123.456.789-09` é um `CPF`, não um `VALOR` (`123.45`) seguido de sobras. O scanner roda os AFDs dos números e fica com o **maior** prefixo aceito (`maior_prefixo`).

**Erros com linha, coluna e dica**, e o scanner **continua** depois deles (modo pânico: descarta só o "pedaço colado" e segue). Dicas do domínio: comando desconhecido (com "você quis dizer"), CPF sem pontuação, horário inválido, protocolo mal formado, vírgula decimal, texto sem fechar aspas...

In [ ]:
# CÉLULA 5 — Mini-scanner feito à mão (linguagem de roteiros do chatbot)
# Cada laço "while" é um ESTADO que fica girando enquanto o caractere pertence ao token.

COMANDOS = {"menu", "opcao", "cpf", "protocolo", "agendar", "msg", "pagar", "ajuda", "voltar", "sair"}   # reservadas
DELIMITADORES = set(" \t\r\n#")           # o que pode vir logo depois de um número

# --- AFDs dos tokens numéricos (os dois primeiros são os validadores do projeto) ---
def classe_hora(c):                        # classes "finas" do Exemplo 4 da aula
    if c in "01":   return "0-1"
    if c == "2":    return "2"
    if c == "3":    return "3"
    if c in "45":   return "4-5"
    if c in "6789": return "6-9"
    if c == ":":    return ":"
    return "outro"
DIG_HORA = ["0-1", "2", "3", "4-5", "6-9"]
delta_hora = {("H0", "0-1"): "H1a", ("H0", "2"): "H1b"}
for d in DIG_HORA:               delta_hora[("H1a", d)] = "H2"
for d in ["0-1", "2", "3"]:      delta_hora[("H1b", d)] = "H2"
delta_hora[("H2", ":")] = "M0"
for d in ["0-1", "2", "3", "4-5"]: delta_hora[("M0", d)] = "M1"
for d in DIG_HORA:               delta_hora[("M1", d)] = "M2"
afd_horario = AFD("Horário HH:MM", delta_hora, "H0", {"M2"}, classe_hora)

def classe_num(c):
    if c in DIGITOS: return "D"
    if c == ".":     return "."
    return "outro"
afd_valor = AFD("VALOR", {("v0", "D"): "v1", ("v1", "D"): "v1", ("v1", "."): "v2",
                          ("v2", "D"): "v3", ("v3", "D"): "v4"}, "v0", {"v4"}, classe_num)      # 150.00
afd_inteiro = AFD("NUM_INT", {("n0", "D"): "n1", ("n1", "D"): "n1"}, "n0", {"n1"}, classe_num)

# ordem = prioridade em caso de empate no tamanho
AFDS_NUMERICOS = [("CPF", afd_cpf), ("PROTOCOLO", afd_protocolo), ("HORARIO", afd_horario),
                  ("VALOR", afd_valor), ("NUM_INT", afd_inteiro)]

def maior_lexema_numerico(fonte, i):
    """MAXIMAL MUNCH: roda todos os AFDs numéricos e fica com o maior prefixo aceito."""
    melhor_tipo, melhor_fim = None, i
    for tipo, afd in AFDS_NUMERICOS:
        fim = afd.maior_prefixo(fonte, i)
        if fim is not None and fim > melhor_fim:
            melhor_tipo, melhor_fim = tipo, fim
    return melhor_tipo, melhor_fim

# --- mensagens de erro (descrição, dica) ---
def dica_numero(bloco):
    """Escolhe a dica olhando o 'bloco' inteiro colado (sem espaços), não só o primeiro caractere ruim."""
    if bloco.count(".") >= 2 or ("." in bloco and "-" in bloco):
        return "CPF tem o formato 000.000.000-00 (11 dígitos, 2 pontos e 1 hífen)"
    if ":" in bloco:
        return "horário tem o formato HH:MM, de 00:00 a 23:59 (ex.: 14:30)"
    if "-" in bloco:
        return "protocolo tem o formato AAAA-NNNNNN, ano 2000 a 2099 e 6 dígitos (ex.: 2026-000123)"
    if "," in bloco:
        return "use ponto como separador decimal (ex.: 150.00)"
    if "." in bloco:
        return "valor monetário tem exatamente 2 casas decimais (ex.: 150.00)"
    return "separe os argumentos com espaço"

def erro_comando(nome):
    """None se o comando existe; senão (descrição, dica)."""
    if nome in COMANDOS:
        return None
    if nome == "":
        return "'/' sozinha", "todo comando é '/' + nome (ex.: /menu, /ajuda)"
    parecido = difflib.get_close_matches(nome.lower(), sorted(COMANDOS), n=1)
    sugestao = f"você quis dizer '/{parecido[0]}'? " if parecido else ""
    return f"comando '/{nome}' não existe", sugestao + "comandos: " + ", ".join("/" + c for c in sorted(COMANDOS))

def fechar_numero(fonte, i, tipo, fim):
    """Confere o lexema numérico. Devolve (token|None, erro|None, novo_i); erro = (posição, descrição, dica)."""
    n, j = len(fonte), i
    while j < n and fonte[j] not in DELIMITADORES:        # j = fim do bloco colado
        j += 1
    if fim == j:                                          # o maior prefixo cobre o bloco todo: lexema válido
        lexema = fonte[i:fim]
        if tipo == "NUM_INT" and len(lexema) == 11:
            return None, (i, f"número de 11 dígitos {lexema!r} sem pontuação", "se for um CPF, use 000.000.000-00"), j
        return (tipo, lexema), None, j
    ch = fonte[fim]                                       # sobrou lixo: o 1º caractere ruim está em 'fim'
    desc = f"número malformado {fonte[i:j]!r}: lido como {tipo} {fonte[i:fim]!r}, mas {ch!r} não cabe"
    return None, (fim, desc, dica_numero(fonte[i:j])), j

def char_invalido(fonte, i):
    """Nenhum estado aceita fonte[i]. Devolve (descrição, dica, novo_i)."""
    c = fonte[i]
    if c == '"':                                          # texto sem aspas de fechamento: descarta até o fim da linha
        j = i
        while j < len(fonte) and fonte[j] != "\n":
            j += 1
        return "texto sem aspas de fechamento", 'feche o texto com " antes do fim da linha', j
    dicas = {"@": "menções (@) não existem; todo comando começa com /",
             "'": 'use aspas duplas (") nos textos',
             "\\": "os comandos começam com / (barra normal)",
             ",": "use ponto como separador decimal (ex.: 150.00)",
             ".": "o ponto só aparece dentro de números (150.00) e CPF (000.000.000-00)",
             ":": "o dois-pontos só aparece em horários (HH:MM)",
             "-": "o hífen só aparece em CPF e protocolo"}
    return f"caractere inesperado {c!r}", dicas.get(c, "caractere fora da linguagem dos roteiros"), i + 1

def escanear(fonte):
    """Devolve (tokens, erros). token = (tipo, lexema, linha, coluna)."""
    tokens, erros = [], []
    n, i, linha, ini_linha = len(fonte), 0, 1, 0

    def col(pos): return pos - ini_linha + 1
    def registrar_erro(pos, desc, dica): erros.append(f"Linha {linha}, coluna {col(pos)}: {desc} (dica: {dica})")

    while i < n:
        c, inicio = fonte[i], i
        if c == "\n":                                     # quebra de linha: só atualiza a posição
            i, linha = i + 1, linha + 1
            ini_linha = i
        elif c in " \t\r":                                # espaço: ignorado
            i += 1
        elif c == "#":                                    # comentário: ignora até o fim da linha
            while i < n and fonte[i] != "\n":
                i += 1
        elif c == "/":                                    # estado COMANDO: "/" + nome; conferimos na lista de reservadas
            i += 1
            while i < n and fonte[i] in ALNUM:
                i += 1
            erro = erro_comando(fonte[inicio + 1:i])
            if erro:
                registrar_erro(inicio, *erro)
            else:
                tokens.append(("COMANDO", fonte[inicio:i], linha, col(inicio)))
        elif c in LETRAS:                                 # estado ID
            while i < n and fonte[i] in ALNUM:
                i += 1
            tokens.append(("ID", fonte[inicio:i], linha, col(inicio)))
        elif c in DIGITOS:                                # família numérica: Maximal Munch entre os AFDs
            tipo, fim = maior_lexema_numerico(fonte, i)
            token, erro, i = fechar_numero(fonte, i, tipo, fim)
            if erro:
                registrar_erro(*erro)
            else:
                tokens.append((token[0], token[1], linha, col(inicio)))
        elif c == '"':                                    # estado TEXTO: até a próxima aspa, na mesma linha
            j = i + 1
            while j < n and fonte[j] not in '"\n':
                j += 1
            if j < n and fonte[j] == '"':
                tokens.append(("TEXTO", fonte[i:j + 1], linha, col(inicio)))
                i = j + 1
            else:
                desc, dica, i = char_invalido(fonte, i)
                registrar_erro(inicio, desc, dica)
        else:                                             # nenhum estado aceita: erro léxico
            desc, dica, i = char_invalido(fonte, i)
            registrar_erro(inicio, desc, dica)
    return tokens, erros

def mostrar_tokens(tokens, erros):
    print(f"{'TIPO':10} {'LEXEMA':26} {'LIN':>3} {'COL':>3}")
    for tipo, lexema, lin, col in tokens:
        print(f"{tipo:10} {lexema:26} {lin:>3} {col:>3}")
    print()
    for e in erros:
        print("⚠️", e)

print("✅ Scanner pronto. Comandos reservados:", ", ".join(sorted(COMANDOS)))

# --- roteiros de exemplo (usados na Célula 6, na interface e no autoteste) ---
ROTEIRO_VALIDO = '''# segunda via de boleto
/menu
/opcao 1
/cpf 123.456.789-09
/protocolo 2026-000123
/pagar 150.00
/ajuda protocolo
/agendar 14:30
/msg "Obrigado pelo atendimento"
/sair
'''
ROTEIRO_COM_ERROS = '''# roteiro cheio de problemas
/menus
/opcao 2 @
/cpf 12345678909
/cpf 123.456.789-0a
/agendar 25:00
/pagar 150,00
/pagar 150.5
/protocolo 2026-12
/protocolo 2026-0001234
/msg "texto sem fechar
/msg "este está ok"
/
/sair
'''

In [ ]:
# CÉLULA 6 — Testando o scanner: um roteiro correto e um roteiro cheio de erros
# (os dois roteiros ROTEIRO_VALIDO e ROTEIRO_COM_ERROS são definidos no fim da Célula 5)

tokens, erros = escanear(ROTEIRO_VALIDO)
print(f"━━━━ Roteiro válido: {len(tokens)} tokens, {len(erros)} erros\n")
mostrar_tokens(tokens, erros)

tokens, erros = escanear(ROTEIRO_COM_ERROS)
print(f"\n━━━━ Roteiro com erros: {len(tokens)} tokens, {len(erros)} erros (o scanner NÃO parou no primeiro)\n")
mostrar_tokens(tokens, erros)

## R6 — Interface: teste tudo sem editar código

Duas abas: **Validadores** (escolha o AFD, digite a palavra, veja o passo a passo e se a RegEx concorda) e **Scanner** (cole um roteiro e veja os tokens e os erros com linha, coluna e dica).

> **Desafio relâmpago:** um colega digita uma palavra "pegadinha" e o outro tenta prever, *antes de clicar*, se será aceita e em qual estado o autômato vai morrer.

In [ ]:
# CÉLULA 7 — Interface (ipywidgets): validadores e scanner
# Esta célula usa o que foi definido nas Células 0 a 5. Se algum nome faltar, a sessão foi reiniciada.
_faltando = [n for n in ("AFD", "afd_cpf", "afd_protocolo", "afd_horario", "REGEX_CPF", "REGEX_PROTOCOLO",
                         "escanear", "mostrar_tokens", "ROTEIRO_COM_ERROS") if n not in globals()]
if _faltando:
    raise RuntimeError(f"Faltam definições {_faltando}: execute as células anteriores antes desta "
                       "(menu Ambiente de execução → Executar tudo).")


AUTOMATOS = {"CPF formatado": (afd_cpf, REGEX_CPF, "123.456.789-09"),
             "Protocolo AAAA-NNNNNN": (afd_protocolo, REGEX_PROTOCOLO, "2026-000123"),
             "Horário HH:MM (usado no scanner)": (afd_horario, r"([01][0-9]|2[0-3]):[0-5][0-9]", "23:59")}

# ---------------- Aba 1: validadores ----------------
dd_val = widgets.Dropdown(options=list(AUTOMATOS), value="CPF formatado", description="Autômato:")
tx_val = widgets.Text(value="123.456.789-09", description="Palavra:")
bt_val = widgets.Button(description="Rodar", button_style="primary")
out_val = widgets.Output()

def ao_validar(_):
    out_val.clear_output()
    with out_val:
        afd, regex, _exemplo = AUTOMATOS[dd_val.value]
        palavra = tx_val.value
        r_afd = afd.rodar(palavra)
        r_re = re.fullmatch(regex, palavra) is not None
        print(f"RegEx {regex!r} {'aceita' if r_re else 'rejeita'} · AFD e RegEx {'concordam ✅' if r_afd == r_re else 'DIVERGEM ⚠️'}")

def ao_trocar_automato(mudanca):
    tx_val.value = AUTOMATOS[mudanca["new"]][2]

bt_val.on_click(ao_validar)
dd_val.observe(ao_trocar_automato, names="value")
aba_validadores = widgets.VBox([widgets.HBox([dd_val, tx_val, bt_val]), out_val])

# ---------------- Aba 2: scanner ----------------
ta_scan = widgets.Textarea(value=ROTEIRO_COM_ERROS, layout=widgets.Layout(width="100%", height="220px"))
bt_scan = widgets.Button(description="Escanear", button_style="primary")
out_scan = widgets.Output()

def ao_escanear(_):
    out_scan.clear_output()
    with out_scan:
        tokens, erros = escanear(ta_scan.value)
        print(f"{len(tokens)} tokens, {len(erros)} erros\n")
        mostrar_tokens(tokens, erros)

bt_scan.on_click(ao_escanear)
aba_scanner = widgets.VBox([bt_scan, ta_scan, out_scan])

abas = widgets.Tab(children=[aba_validadores, aba_scanner])
for k, titulo in enumerate(["🔎 Validadores", "🔤 Scanner"]):
    abas.set_title(k, titulo)
display(abas)
ao_validar(None); ao_escanear(None)       # já mostra um resultado ao abrir

## Autoteste final — o checklist de entrega em uma tabela

In [ ]:
# CÉLULA 8 — Autoteste final: confere cada item do checklist de entrega

def checar(descricao, funcao):
    try:
        ok = bool(funcao())
    except Exception as e:
        ok, descricao = False, f"{descricao} (exceção: {type(e).__name__}: {e})"
    return ("✅" if ok else "❌", descricao)

tokens_ok, erros_ok = escanear(ROTEIRO_VALIDO)
tokens_er, erros_er = escanear(ROTEIRO_COM_ERROS)
tipos_de_token = {t[0] for t in tokens_ok}
dicas_usadas = {e.split("(dica: ")[1].split(")")[0][:25] for e in erros_er}

resultados = [
    checar("R1: 2 validadores com 5+ estados cada", lambda: len(afd_cpf.estados) >= 5 and len(afd_protocolo.estados) >= 5),
    checar("R1: um validador com classes finas (protocolo: 0, 2 e 1,3-9)", lambda: {"0", "2", "1,3-9"} <= set(afd_protocolo.alfabeto)),
    checar("R2: 1 palavra aceita e 1 rejeitada por AFD (rodar)", lambda: afd_cpf.aceita("123.456.789-09") and not afd_cpf.aceita("123.456.789-0")
                                                                   and afd_protocolo.aceita("2026-000123") and not afd_protocolo.aceita("2126-000123")),
    checar("R3: 5+ aceitas e 5+ rejeitadas por AFD, com a palavra vazia",
           lambda: all(len(a) >= 5 and len(r) >= 5 and "" in r for _, _, _, a, r in BATERIAS)),
    checar("R3: 0 divergências AFD × RegEx (listas + 5000 palavras aleatórias por AFD)", lambda: DIVERGENCIAS_TOTAIS == 0),
    checar("R4: 5+ estados e 6+ transições", lambda: len(conversa.estados) >= 5 and len(delta_chat) >= 6),
    checar("R4: 2 fluxos válidos finalizam", lambda: processar_conversa(["menu", "opcao_1", "cpf", "protocolo", "pagar", "sair"]) == "finalizada"
                                                   and processar_conversa(["menu", "opcao_2", "msg", "agendar", "voltar", "ajuda", "sair"]) == "finalizada"),
    checar("R4: 2 fluxos inválidos são recusados (com os eventos permitidos)", lambda: processar_conversa(["menu", "protocolo"]) == "recusada"
                                                                                      and processar_conversa(["menu", "sair", "menu"]) == "recusada"),
    checar("R5: 8+ tipos de token", lambda: len(tipos_de_token | {t[0] for t in tokens_er}) >= 8),
    checar("R5: 3+ palavras reservadas", lambda: len(COMANDOS) >= 3),
    checar("R5: Maximal Munch (2026-000123 = 1 PROTOCOLO; 14:30 = 1 HORARIO; CPF = 1 token)",
           lambda: [t[0] for t in escanear("2026-000123 14:30 123.456.789-09")[0]] == ["PROTOCOLO", "HORARIO", "CPF"]),
    checar("R5: espaços e comentários são ignorados", lambda: [t[0] for t in escanear("  # nada\n  /sair # fim")[0]] == ["COMANDO"]),
    checar("R5: erros com linha, coluna e dica", lambda: len(erros_er) >= 5 and all(re.match(r"Linha \d+, coluna \d+: .*\(dica: .+\)$", e) for e in erros_er)),
    checar("R5: 2+ dicas específicas do domínio", lambda: len(dicas_usadas) >= 2),
    checar("R5: o scanner continua depois do erro (tokens depois do 1º erro)", lambda: tokens_er[-1][1] == "/sair"),
    checar("R6: interface montada (2 abas)", lambda: len(abas.children) == 2),
]
print(f"🔬 {sum(r[0] == '✅' for r in resultados)}/{len(resultados)} verificações passaram\n")
for marca, descricao in resultados:
    print(marca, descricao)